# PURITY Training (clean)

Minimal training pipeline for `PURITYHybridModel`. Losses and helpers live in `unified_reco.train_utils`.


## 1. Imports

In [1]:
import os
import sys
import torch
from torch_geometric.loader import DataLoader
from tqdm import tqdm

sys.path.append(os.path.abspath('..'))

from unified_reco.models import PURITYHybridModel
from unified_reco.dataset import PURITYDataset
from unified_reco.train_utils import PURITYLoss, format_targets_from_batch


/home/omar/miniconda3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config

In [3]:
#DATA_DIR = '/mnt/c/Users/obbee/research/notebooks/ML/data/purity/mixed_data_75k.parquet'
DATA_DIR = '/mnt/c/Users/obbee/research/notebooks/ML/data/purity/mixed_data_20k_val.parquet'
EPOCHS = 20
BATCH_SIZE = 12
MAX_HITS = 210

task_weights = {
    'w_atar_slice_multi':   0.2,
    'w_node_pdg':           1.5,
    'w_slice_pdg':          1.0,
    'w_atar_trigger_slice': 0.5,
    'w_pion_kinematics':    0.25,
    'w_endpoints':          0.025,
    'w_positron_angle':     0.5,
    'w_lyso_condensation':  0.25,
    'w_event_builder':      0.1,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cuda


## 3. Data

In [4]:
dataset = PURITYDataset(DATA_DIR, max_hits=MAX_HITS)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Events: {len(dataset)}  Batches/epoch: {len(dataloader)}")


Loading merged parquet dataset from /mnt/c/Users/obbee/research/notebooks/ML/data/purity/mixed_data_20k_val.parquet...
Dropped 191 events exceeding 210 hits or containing 0 hits. Active dataset size: 19809
Events: 19809  Batches/epoch: 1651


## 4. Model

In [8]:
model = PURITYHybridModel()

# Optional: load checkpoint
#model = PURITYHybridModel()
model.load_state_dict(torch.load('/mnt/c/Users/obbee/research/notebooks/ML/model_weights/PURITY_basic_4_12_2026.pth'), strict=False)
model.to(device)


PURITYHybridModel(
  (atar_feature_proj): Sequential(
    (0): Linear(in_features=3, out_features=150, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=150, out_features=150, bias=True)
    (3): LayerNorm((150,), eps=1e-05, elementwise_affine=True)
  )
  (atar_view_embedding): Embedding(2, 150)
  (lyso_encoder): Sequential(
    (0): Linear(in_features=5, out_features=150, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=150, out_features=150, bias=True)
    (3): LayerNorm((150,), eps=1e-05, elementwise_affine=True)
  )
  (atar_blocks): ModuleList(
    (0-2): 3 x JointAttentionBlock(
      (ln1): LayerNorm((150,), eps=1e-05, elementwise_affine=True)
      (conv): TransformerConv(150, 30, heads=5)
      (ln2): LayerNorm((150,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=150, out_features=600, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.05, inplace=False)
       

## 5. Loss + Optimizer

Attention-bias scalars (σ parameters) get a higher learning rate so they can adapt from their physics-motivated initial values; everything else is trained at 1e-4.

In [9]:
criterion = PURITYLoss(config=task_weights)

bias_scalar_params = [
    model.sigma_t_atar_ns,
    model.sigma_t_lyso_floor_ns,
    model.sigma_t_lyso_scale_ns,
    model.angle_sigma_floor,
    model.angle_sigma_scale,
]
bias_ids = {id(p) for p in bias_scalar_params}

event_params = (
    list(model.slim_event_transformer.parameters()) +
    list(model.lyso_event_proj.parameters()) +
    list(model.atar_event_down.parameters()) +
    list(model.event_head.parameters()) +
    list(model.event_modality_emb.parameters()) +
    list(model.event_slice_emb.parameters())
)
event_params = [p for p in event_params if id(p) not in bias_ids]
event_ids = {id(p) for p in event_params}

base_params = [p for p in model.parameters()
               if id(p) not in event_ids and id(p) not in bias_ids]

optimizer = torch.optim.Adam([
    {'params': base_params,        'lr': 1e-4, 'weight_decay': 1e-8},
    {'params': event_params,       'lr': 5e-4},
    {'params': bias_scalar_params, 'lr': 1e-3, 'weight_decay': 0.0},
])


## 6. Training loop

In [10]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    epoch_loss_dict = {}

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        outputs = model(batch.x, batch.batch, task_weights=task_weights)
        targets = format_targets_from_batch(batch)
        loss, loss_dict = criterion(outputs, targets, batch=batch)

        if (not isinstance(loss, torch.Tensor) or not loss.requires_grad
                or torch.isnan(loss) or torch.isinf(loss)):
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()

        total_loss += loss.item()
        for k, v in loss_dict.items():
            if k == 'loss_total':
                continue
            val = v.item() if hasattr(v, 'item') else v
            epoch_loss_dict[k] = epoch_loss_dict.get(k, 0.0) + val

        log_str = " | ".join(
            f"{k.split('_')[-1]}: {(v.item() if hasattr(v, 'item') else v):.4f}"
            for k, v in loss_dict.items() if k != 'loss_total'
        )
        pbar.set_postfix_str(f"Loss: {loss.item():.5f} | {log_str}")

    n = len(dataloader)
    avg_str = " | ".join(f"{k.split('_')[-1]}: {v/n:.4f}" for k, v in epoch_loss_dict.items())
    print(f"Epoch {epoch+1} | Avg loss: {total_loss/n:.4f} | {avg_str}")


Epoch 1/20: 100%|██████████| 1651/1651 [08:48<00:00,  3.12it/s, Loss: -0.13157 | multi: 0.0000 | pdg: 0.0041 | pdg: 0.0401 | slice: 0.0028 | kinematics: 0.0003 | PosLoss: -7.8755 | SpanLoss: 0.0122 | DirLoss: 0.0008 | MeanWidth: 0.0876 | MeanError: 0.1101 | angle: 0.0181 | condensation: 0.0142 | beta: 0.0048 | potential: 0.0094 | fraction: 0.0000 | builder: 0.0485]


Epoch 1 | Avg loss: -0.0845 | multi: 0.0052 | pdg: 0.0075 | pdg: 0.0153 | slice: 0.0247 | kinematics: 0.0009 | PosLoss: -7.0311 | SpanLoss: 0.4372 | DirLoss: 0.0058 | MeanWidth: 0.1688 | MeanError: 0.2638 | angle: 0.0260 | condensation: 0.0843 | beta: 0.0396 | potential: 0.0418 | fraction: 0.0030 | builder: 0.1587


Epoch 2/20:   2%|▏         | 30/1651 [00:08<07:18,  3.70it/s, Loss: -0.09861 | multi: 0.0001 | pdg: 0.0050 | pdg: 0.0033 | slice: 0.0694 | kinematics: 0.0002 | PosLoss: -7.5725 | SpanLoss: 0.0784 | DirLoss: 0.0037 | MeanWidth: 0.2300 | MeanError: 0.3767 | angle: 0.0325 | condensation: 0.0508 | beta: 0.0172 | potential: 0.0280 | fraction: 0.0056 | builder: 0.1587]


KeyboardInterrupt: 

## 7. Save

In [ ]:
# torch.save(model.state_dict(),
#   '/mnt/c/Users/obbee/research/notebooks/ML/model_weights/PURITY_<tag>.pth')
